# Copilot Audit Log - Processor  --  Value Lens 2307 (Spark) update

**Purpose.** Move the heavy Power Query transformations off the Power BI refresh path.
This notebook does, once in Spark, everything the `Chat + Agent Interactions (Audit Logs)`
Power Query used to do row-by-row in the single-threaded mashup engine:

* parse `AccessedResources` / `AISystemPlugin` JSON and flatten them,
* explode the accessed-resources list (1 interaction -> N resource rows),
* derive `InteractionDate` / `WeekStart` / `MonthStart`,
* normalise the UPN and left-join the licence flag,
* resolve `Agent_LinkID` via the 3-way agent map (Entra id -> Title id -> name).

It writes a single, flat, V-Ordered Delta table **`copilot_interactions_curated`** whose
column set is identical to the old Power Query output, so every calculated column,
measure and relationship in the model keeps working unchanged.

Power BI then reads this table with **no transformation** (thin, foldable passthrough),
which is what makes both Direct Lake and a fast Incremental Refresh possible.

> Run this AFTER the audit-log ingester has produced `copilot_interactions_parsed`,
> and after the licensed-users and agents-365 producers have landed their tables.
> Schedule it in the same pipeline, immediately before the semantic-model refresh.

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SRC_INTERACTIONS = "copilot_interactions_parsed"   # raw fact from the audit-log ingester
SRC_LICENSED     = "copilot_licensed_users"        # licensed-users dim (optional)
SRC_AGENTS       = "agents_365"                    # agents 365 dim (optional)

OUT_TABLE        = "copilot_interactions_curated"  # <-- Power BI reads this

# Full rebuild vs incremental append. For the initial backfill use "overwrite".
# For daily runs use "merge" (idempotent upsert on the natural key below).
WRITE_MODE       = "overwrite"                       # "overwrite" | "merge"
MERGE_KEYS       = ["Id"]                            # unique interaction id column(s) if present

RUN_OPTIMIZE     = True                              # OPTIMIZE + VORDER after write

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import DataFrame

# V-Order every Parquet/Delta file this session writes (required for Direct Lake perf).
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
# Let Spark evolve the Delta schema when tenant exports add/drop optional columns.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


def has_col(df: DataFrame, name: str) -> bool:
    return name in df.columns


def ensure_col(df: DataFrame, name: str, default=F.lit(None), dtype="string") -> DataFrame:
    # Idempotent 'add column if missing' -- mirrors the M Table.HasColumns guards.
    if has_col(df, name):
        return df
    return df.withColumn(name, default.cast(dtype))


def first_existing(df: DataFrame, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback

In [ ]:
# ----------------------------------------------------------------------------
# 1. READ RAW FACT
# ----------------------------------------------------------------------------
fact = spark.table(SRC_INTERACTIONS)
print(f"{SRC_INTERACTIONS}: {fact.count():,} rows, {len(fact.columns)} cols")

In [ ]:
# ----------------------------------------------------------------------------
# 2. ENSURE OPTIONAL COLUMNS EXIST (tenant exports vary)
#    Mirrors the 'Add ...' idempotent steps in the Power Query.
# ----------------------------------------------------------------------------
fact = ensure_col(fact, "Message_isPrompt", F.lit("TRUE"))
fact = ensure_col(fact, "ModelTransparencyDetails_ModelProviderName")
fact = ensure_col(fact, "ModelTransparencyDetails_ModelName")
fact = ensure_col(fact, "ApplicationName")
fact = ensure_col(fact, "Audit_UserKey")
fact = ensure_col(fact, "SensitivityLabelId")
fact = ensure_col(fact, "AccessedResource_SensitivityLabelId")

# AppIdentity -> AppIdentity_AppId / AppIdentity_DisplayName, then drop the raw column.
fact = ensure_col(fact, "AppIdentity_AppId")
if has_col(fact, "AppIdentity"):
    if not has_col(fact, "AppIdentity_DisplayName"):
        fact = fact.withColumn("AppIdentity_DisplayName", F.col("AppIdentity").cast("string"))
    fact = fact.drop("AppIdentity")
else:
    fact = ensure_col(fact, "AppIdentity_DisplayName")

In [ ]:
# ----------------------------------------------------------------------------
# 3. ACCESSED RESOURCES  --  parse JSON + explode (this was the big fold-breaker)
#    Old M: Json.Document -> Table.ExpandListColumn -> Table.ExpandRecordColumn.
#    Here: from_json to an array<struct>, explode_outer, then project.
# ----------------------------------------------------------------------------
ar_schema = T.ArrayType(T.StructType([
    T.StructField("Type",    T.StringType()),
    T.StructField("Action",  T.StringType()),
    T.StructField("SiteUrl", T.StringType()),
]))

if has_col(fact, "AccessedResources"):
    parsed = fact.withColumn(
        "_resources",
        F.when(
            (F.col("AccessedResources").isNull()) | (F.trim(F.col("AccessedResources")) == ""),
            F.array().cast(ar_schema),
        ).otherwise(F.coalesce(F.from_json(F.col("AccessedResources"), ar_schema),
                               F.array().cast(ar_schema))),
    ).drop("AccessedResources")

    # explode_outer keeps interactions that have zero accessed resources (LEFT semantics).
    parsed = parsed.withColumn("_res", F.explode_outer("_resources")).drop("_resources")
    fact = (parsed
            .withColumn("AccessedResource_Type",    F.col("_res.Type"))
            .withColumn("AccessedResource_Action",  F.col("_res.Action"))
            .withColumn("AccessedResource_SiteUrl", F.col("_res.SiteUrl"))
            .drop("_res"))
else:
    fact = ensure_col(fact, "AccessedResource_Type")
    fact = ensure_col(fact, "AccessedResource_Action")
    fact = ensure_col(fact, "AccessedResource_SiteUrl")

In [ ]:
# ----------------------------------------------------------------------------
# 4. AI SYSTEM PLUGIN  --  parse JSON, take first element if it is a list
# ----------------------------------------------------------------------------
plugin_obj = T.StructType([
    T.StructField("Id",   T.StringType()),
    T.StructField("Name", T.StringType()),
])
plugin_arr = T.ArrayType(plugin_obj)

if has_col(fact, "AISystemPlugin"):
    raw = F.col("AISystemPlugin")
    as_arr = F.from_json(raw, plugin_arr)
    as_obj = F.from_json(raw, plugin_obj)
    single = F.when(as_arr.isNotNull() & (F.size(as_arr) > 0), as_arr.getItem(0)).otherwise(as_obj)
    fact = (fact
            .withColumn("_plugin", F.when((raw.isNull()) | (F.trim(raw) == ""), F.lit(None).cast(plugin_obj)).otherwise(single))
            .withColumn("AISystemPlugin_Id",   F.col("_plugin.Id"))
            .withColumn("AISystemPlugin_Name", F.col("_plugin.Name"))
            .drop("_plugin").drop("AISystemPlugin"))
else:
    fact = ensure_col(fact, "AISystemPlugin_Id")
    fact = ensure_col(fact, "AISystemPlugin_Name")

In [ ]:
# ----------------------------------------------------------------------------
# 5. DATES + RESOURCE COUNT
# ----------------------------------------------------------------------------
fact = fact.withColumn("CreationDate", F.to_timestamp("CreationDate"))

for c in ["InteractionDate", "WeekStart", "MonthStart"]:
    if has_col(fact, c):
        fact = fact.drop(c)

fact = (fact
        .withColumn("InteractionDate", F.to_date("CreationDate"))
        # Power Query used Day.Monday as the week start.
        .withColumn("WeekStart", F.date_sub(F.next_day(F.col("InteractionDate"), "Mon"), 7))
        .withColumn("MonthStart", F.trunc(F.col("InteractionDate"), "month")))

if has_col(fact, "Resource_Count"):
    fact = fact.withColumn("Resource_Count", F.col("Resource_Count").cast("long"))
else:
    fact = fact.withColumn("Resource_Count", F.lit(1).cast("long"))

In [ ]:
# ----------------------------------------------------------------------------
# 6. NORMALISED UPN  (join key to the licensed-users dim)
# ----------------------------------------------------------------------------
if has_col(fact, "Audit_UserId_Normalized"):
    norm = F.coalesce(F.col("Audit_UserId_Normalized"),
                      F.lower(F.trim(F.col("Audit_UserId").cast("string"))))
elif has_col(fact, "Audit_UserId"):
    norm = F.lower(F.trim(F.col("Audit_UserId").cast("string")))
else:
    norm = F.lit(None).cast("string")
fact = fact.withColumn("_NormUPN", norm)

In [ ]:
# ----------------------------------------------------------------------------
# 7. LEFT-JOIN LICENCE FLAG  (dedup dim first -> no row fan-out)
# ----------------------------------------------------------------------------
def load_licensed():
    try:
        lic = spark.table(SRC_LICENSED)
    except Exception as e:
        print(f"[warn] {SRC_LICENSED} not found ({e}); 'Has license' will be null.")
        return None
    upn_col = first_existing(lic, ["User Principal Name", "userPrincipalName",
                                   "UserPrincipalName", "User principal name",
                                   "User_Principal_Name"])
    has_lic = first_existing(lic, ["Has license", "Has License", "HasLicense", "HasCopilot",
                                   "Has Copilot", "Has Copilot License", "HasCopilotLicense",
                                   "isUser", "Has_license"])
    if has_col(lic, "UPN_Normalized"):
        key = F.col("UPN_Normalized")
    elif upn_col:
        key = F.lower(F.trim(F.col(upn_col).cast("string")))
    else:
        return None
    hl = F.col(has_lic).cast("string") if has_lic else F.lit("Unknown")
    lic = (lic.withColumn("UPN_Normalized", key)
              .withColumn("Has license", hl)
              .where(F.col("UPN_Normalized").isNotNull() & (F.trim("UPN_Normalized") != ""))
              .select("UPN_Normalized", "Has license")
              .dropDuplicates(["UPN_Normalized"]))
    return lic

lic = load_licensed()
if lic is not None:
    fact = fact.join(F.broadcast(lic), fact["_NormUPN"] == lic["UPN_Normalized"], "left") \
               .drop(lic["UPN_Normalized"])
else:
    fact = ensure_col(fact, "Has license")

In [ ]:
# ----------------------------------------------------------------------------
# 8. AGENT_TITLEID  (keep existing; else derive text before first '.' of AgentId)
# ----------------------------------------------------------------------------
aid = F.trim(F.col("AgentId").cast("string")) if has_col(fact, "AgentId") else F.lit(None).cast("string")
derived = F.when(aid.isNull() | (aid == ""), F.lit(None)) \
           .otherwise(F.when(F.instr(aid, ".") > 0, F.substring_index(aid, ".", 1)).otherwise(aid))
if has_col(fact, "Agent_TitleID"):
    existing = F.trim(F.col("Agent_TitleID").cast("string"))
    fact = fact.withColumn("Agent_TitleID",
                           F.when(existing.isNotNull() & (existing != ""), existing).otherwise(derived))
else:
    fact = fact.withColumn("Agent_TitleID", derived)

fact = ensure_col(fact, "Agent_EntraId")
fact = ensure_col(fact, "AgentName")

In [ ]:
# ----------------------------------------------------------------------------
# 9. THREE DEDUPED AGENT MAPS  (Entra id / Title id / normalised name -> Title ID)
#    then resolve Agent_LinkID = first non-null of Entra, Direct, Name.
# ----------------------------------------------------------------------------
def load_agents():
    try:
        return spark.table(SRC_AGENTS)
    except Exception as e:
        print(f"[warn] {SRC_AGENTS} not found ({e}); Agent_LinkID falls back to null.")
        return None

ag = load_agents()
fact = fact.withColumn("__nkey_fact", F.lower(F.trim(F.col("AgentName").cast("string"))))

if ag is not None and "Title ID" in ag.columns:
    entra_col = first_existing(ag, ["Entra Agent ID", "EntraAgentId", "Entra Agent Id",
                                    "EntraAgentID", "Agent ID", "AgentId", "Agent Id",
                                    "Bot Id", "BotId"])
    name_col  = first_existing(ag, ["Agent name", "Name"])

    title = F.trim(F.col("Title ID").cast("string"))
    ag2 = ag.withColumn("_title", title)

    entra_map = (ag2.withColumn("_entra", F.trim(F.col(entra_col).cast("string")) if entra_col else F.lit(None).cast("string"))
                    .where(F.col("_entra").isNotNull() & (F.col("_entra") != ""))
                    .select("_entra", "_title").dropDuplicates(["_entra"])) if entra_col else None
    title_map = (ag2.where(F.col("_title").isNotNull() & (F.col("_title") != ""))
                    .select(F.col("_title").alias("_tkey"), F.col("_title").alias("_title2"))
                    .dropDuplicates(["_tkey"]))
    name_map  = (ag2.withColumn("_nkey", F.lower(F.trim(F.col(name_col).cast("string"))) if name_col else F.lit(None).cast("string"))
                    .where(F.col("_nkey").isNotNull() & (F.col("_nkey") != ""))
                    .select("_nkey", "_title").dropDuplicates(["_nkey"])) if name_col else None

    if entra_map is not None:
        fact = fact.join(F.broadcast(entra_map), fact["Agent_EntraId"] == entra_map["_entra"], "left") \
                   .withColumnRenamed("_title", "__EntraTitle").drop("_entra")
    else:
        fact = fact.withColumn("__EntraTitle", F.lit(None).cast("string"))

    fact = fact.join(F.broadcast(title_map), fact["Agent_TitleID"] == title_map["_tkey"], "left") \
               .withColumnRenamed("_title2", "__DirectTitle").drop("_tkey")

    if name_map is not None:
        fact = fact.join(F.broadcast(name_map), fact["__nkey_fact"] == name_map["_nkey"], "left") \
                   .withColumnRenamed("_title", "__NameTitle").drop("_nkey")
    else:
        fact = fact.withColumn("__NameTitle", F.lit(None).cast("string"))
else:
    fact = fact.withColumn("__EntraTitle", F.lit(None).cast("string")) \
               .withColumn("__DirectTitle", F.lit(None).cast("string")) \
               .withColumn("__NameTitle", F.lit(None).cast("string"))

def _clean(c):
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNotNull() & (t != ""), t)

fact = fact.withColumn("Agent_LinkID",
                       F.coalesce(_clean("__EntraTitle"), _clean("__DirectTitle"), _clean("__NameTitle")))
fact = fact.drop("__EntraTitle", "__DirectTitle", "__NameTitle", "__nkey_fact",
                 "_NormUPN", "Audit_UserId_Normalized")

In [ ]:
# ----------------------------------------------------------------------------
# 10. GUARANTEE THE MODEL CONTRACT
#     The semantic model binds to these 33 sourced columns. A tenant export can
#     omit optional ones, so add any missing as typed nulls -- the import then
#     never errors 'column not found', exactly like the old Power Query guards.
# ----------------------------------------------------------------------------
REQUIRED_TEXT_COLS = [
    "AISystemPlugin_Id", "AISystemPlugin_Name",
    "AccessedResource_Action", "AccessedResource_SensitivityLabelId",
    "AccessedResource_SiteUrl", "AccessedResource_Type",
    "AgentId", "AgentName", "Agent_EntraId", "Agent_LinkID", "Agent_TitleID",
    "AppHost", "AppIdentity_AppId", "AppIdentity_DisplayName", "AppIdentity_PublisherId",
    "ApplicationName", "Audit_UserId", "Audit_UserKey", "ClientRegion", "Context_Type",
    "Has license", "Message_Id", "Message_isPrompt",
    "ModelTransparencyDetails_ModelName", "ModelTransparencyDetails_ModelProviderName",
    "SensitivityLabelId", "ThreadId", "Workload",
]
for c in REQUIRED_TEXT_COLS:
    fact = ensure_col(fact, c)                    # text, null when absent

# Non-text contract columns are always produced above, but keep types explicit.
fact = fact.withColumn("CreationDate", F.col("CreationDate").cast("timestamp")) \
           .withColumn("InteractionDate", F.col("InteractionDate").cast("date")) \
           .withColumn("WeekStart", F.col("WeekStart").cast("date")) \
           .withColumn("MonthStart", F.col("MonthStart").cast("date")) \
           .withColumn("Resource_Count", F.col("Resource_Count").cast("long"))

missing = [c for c in (REQUIRED_TEXT_COLS + ["CreationDate","InteractionDate","WeekStart","MonthStart","Resource_Count"]) if c not in fact.columns]
assert not missing, f"model contract broken, missing: {missing}"
print("model contract satisfied: all 33 sourced columns present")

In [ ]:
# ----------------------------------------------------------------------------
# 11. WRITE CURATED DELTA  (Power BI reads this table verbatim)
# ----------------------------------------------------------------------------
print(f"curated: {len(fact.columns)} cols -> {OUT_TABLE}  (mode={WRITE_MODE})")

if WRITE_MODE == "merge" and spark.catalog.tableExists(OUT_TABLE) and all(k in fact.columns for k in MERGE_KEYS):
    from delta.tables import DeltaTable
    tgt = DeltaTable.forName(spark, OUT_TABLE)
    cond = " AND ".join([f"t.`{k}` = s.`{k}`" for k in MERGE_KEYS])
    (tgt.alias("t").merge(fact.alias("s"), cond)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    (fact.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .format("delta").saveAsTable(OUT_TABLE))

if RUN_OPTIMIZE:
    spark.sql(f"OPTIMIZE {OUT_TABLE} VORDER")   # compact + V-Order for Direct Lake / fast import

print("done. row count:", spark.table(OUT_TABLE).count())

## Point Power BI at the curated table

In the template, the `Chat + Agent Interactions (Audit Logs)` partition M becomes a thin,
**foldable** passthrough (no JSON parse, no expand, no joins):

```m
let
    Source   = FabricTable("copilot_interactions_curated"),
    Filtered = Table.SelectRows(Source, each [CreationDate] >= RangeStart and [CreationDate] < RangeEnd)
in
    Filtered
```

Because the only step is a range filter on `CreationDate`, it folds to the Lakehouse SQL
endpoint and Incremental Refresh only touches new day-partitions. On Fabric / Premium
capacity you can instead flip the model to **Direct Lake** and skip refresh entirely.